# Part 14, notebook 1 of 6: the desk and the library

Companion to *Context Engineering Starts With Retrieval*. The six notebooks follow the post in order, explain each idea with a daily-life picture, show the real source, and run it.

| # | Notebook | Post section | Needs the model server? |
|---|---|---|---|
| 14a | `14a_desk_and_archive` | The desk and the library, the archive | no |
| 14b | `14b_harness_switches` | The harness: five ideas, five switches | **yes** (3 live runs, about 15 s) |
| 14c | `14c_results_t1_to_t5` | Results, T1 to T5 | no (reads the saved runs) |
| 14d | `14d_subagent_brief` | T6: how much should a sub-agent be given? | **yes** (2 live calls) |
| 14e | `14e_two_frameworks` | Deep Agents and Google ADK | no (reads the saved runs) |
| 14f | `14f_loops_guard_sandbox` | What I found: loops, the repeat guard, the sandbox | Docker only |

**The picture.** You are doing an open-book exam. The library has ten thousand books and your desk fits three. The skill is not knowing everything, it is fetching the right three pages. Things go wrong in three ways: you look in the wrong place, you drag back a whole encyclopedia when you needed a paragraph, or you fetch an old edition that contradicts the new one.

In agent terms: the **context window** is the desk (this server's is 65,536 tokens), the **harness** is the librarian's rules, and everything on the desk is re-sent on every step of the loop.

In [1]:
import sys, json, os
from pathlib import Path
os.environ.setdefault("AGENTIC_ZERO_MODEL", "qwen3.8-27b")   # the model name the local vLLM server serves (as in the post)
os.environ.setdefault("RUST_LOG", "error"); os.environ.setdefault("LANCE_LOG", "error")   # silence LanceDB deprecation chatter
import warnings; warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve()))                # this notebook lives in literate/, the code in notebooks/agentic-zero-to-advanced/
OUT = Path("..").resolve() / "out"
from ctx_corpus import ARCHIVE
from ctx_harness import est_tokens
WINDOW = 65_536
print("archive folder:", ARCHIVE.name, "| window:", f"{WINDOW:,}", "tokens")

archive folder: ctx_archive | window: 65,536 tokens


## What is in the archive

Real files from this series: the 4,000-row table from the synthetic void dataset, the 200 reports that the Part 7 pipeline wrote, and two policy files. Token counts below are **characters divided by four**, a rough estimate; the server's own count is a little higher (it counted 28,300 for the reports with a question attached).

In [2]:
rows = []
for folder in ("exports", "reports", "policy", "cases"):
    files = [f for f in (ARCHIVE / folder).rglob("*") if f.is_file()]
    chars = sum(len(f.read_text()) for f in files)
    rows.append((folder, len(files), chars, est_tokens("x" * chars)))
print(f"{'folder':10s} {'files':>6s} {'KB':>8s} {'~tokens':>9s}  fits in the {WINDOW:,}-token window?")
for folder, n, chars, tok in rows:
    print(f"{folder:10s} {n:6d} {chars/1000:8.0f} {tok:9,d}  {'yes' if tok < WINDOW else 'NO, it is ' + format(tok / WINDOW, '.1f') + 'x the window'}")

folder      files       KB   ~tokens  fits in the 65,536-token window?
exports         1      450   112,500  NO, it is 1.7x the window
reports       200       91    22,806  yes
policy          2        1       233  yes
cases         600      143    35,745  yes


This single table explains most of the post. The table (`exports`) does **not** fit, so something has to fetch from it. The 200 reports **do** fit, so pasting them in is a legitimate strategy, and in the results it beats the clever one on two of the tasks.

## One record of each kind

In [3]:
print("--- exports/inspection_log.csv (first 3 lines)")
print("\n".join((ARCHIVE / "exports" / "inspection_log.csv").read_text().splitlines()[:3]))
print("\n--- reports/test_00000.md")
print((ARCHIVE / "reports" / "test_00000.md").read_text())

--- exports/inspection_log.csv (first 3 lines)
id,split,src_index,part_class,part_name,mu,blur,noise,gain,grad,grad_angle,cluster,n_voids,void_area,part_area,max_eq_diam,min_contrast
train_00000,train,23110,5,Sandal,2.75859,1.20895,0.04829,0.97696,0.04378,4.25449,False,1,51,7083,8.05824,0.14802
train_00001,train,479,9,Ankle boot,2.10485,1.47234,0.02143,0.93257,0.07269,4.90637,False,3,298,9525,16.92569,0.05649

--- reports/test_00000.md
# Report test_00000

verdict: pass
candidates_examined: 3
candidates_kept: 0
reason: no candidate survived the classifier and the rules

- rejected candidate 0 (by classifier)
- rejected candidate 1 (by classifier)
- rejected candidate 2 (by classifier)

## Note
Pass. 3 candidate(s) examined, none kept. no candidate survived the classifier and the rules.



## The two policy files, and the one I planted

`policy-2026-10-17.md` holds the real constants from the pipeline. `policy-2026-09-12.md` is **planted by me**: a "superseded" policy with a different minimum area (35 px instead of 20). It exists only to test whether an agent reads a stale document. It is not a real earlier policy.

In [4]:
import difflib
a = (ARCHIVE / "policy" / "policy-2026-09-12.md").read_text().splitlines()
b = (ARCHIVE / "policy" / "policy-2026-10-17.md").read_text().splitlines()
print("\n".join(l for l in difflib.unified_diff(a, b, "policy-2026-09-12.md (planted, stale)", "policy-2026-10-17.md (current)", lineterm="", n=0)))

--- policy-2026-09-12.md (planted, stale)
+++ policy-2026-10-17.md (current)
@@ -1 +1 @@
-# Void reporting policy (2026-09-12)
+# Void reporting policy (2026-10-17)
@@ -3 +3 @@
-status: SUPERSEDED by policy-2026-10-17
+status: current
@@ -5 +5 @@
-- Minimum reportable void area: 35 px. Smaller boxes are not reported.
+- Minimum reportable void area: 20 px. Smaller boxes are not reported.


## How the archive is built

`ctx_corpus.py` is short. The reports are rendered from the pipeline's saved results, one markdown file per image.

In [5]:
import inspect, ctx_corpus
print(inspect.getsource(ctx_corpus.report_markdown))

def report_markdown(r: dict) -> str:
    rep = r["report"]
    lines = [f"# Report {r['image']}", "",
             f"verdict: {rep['verdict']}",
             f"candidates_examined: {r['n_candidates']}",
             f"candidates_kept: {r['kept']}",
             f"reason: {rep['reason']}", ""]
    for k in rep["kept"]:
        lines.append(f"- kept candidate {k['id']}: severity {k['severity']}, confidence {k['confidence']}, "
                     f"uncertain {str(k['uncertain']).lower()}, contrast {k['contrast']}, area_px {k['area_px']}")
    for k in rep["rejected"]:
        lines.append(f"- rejected candidate {k['id']} (by {k['by']})")
    lines += ["", "## Note", rep["summary"], ""]
    return "\n".join(lines)



**Next:** notebook 14b turns the five retrieval ideas into switches and runs the harness on this archive.